# MindBridge.ai — Matching Engine Demo

End-to-end walkthrough of the hybrid matching engine on the committed sample data.

- **Stage 1:** semantic retrieval (sentence-transformers, or TF-IDF fallback offline)
- **Stage 2:** heuristic reranker → final score + human-readable *reasons*

Run top-to-bottom. No API keys needed.

In [ ]:
# If running from the notebooks/ folder, hop up to the repo root so imports resolve.
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

from mindbridge.ingestion.registry import load_jobs, load_candidates
from mindbridge.matching.engine import MatchEngine

jobs = load_jobs(sources=['sample'])
candidates = load_candidates(sources=['sample'])
print(f'{len(jobs)} jobs, {len(candidates)} candidates loaded')

In [ ]:
# Build the engine. First run may download the embedding model (~90MB);
# if offline it transparently falls back to TF-IDF.
engine = MatchEngine()
print('embedder backend :', engine.embedder_backend)
print('reranker backend :', engine.reranker_backend)

## Hiree flow — best jobs for a candidate

In [ ]:
import pandas as pd

def show(results):
    return pd.DataFrame([
        {'match': r.matched_label, 'score': round(r.score, 3),
         'semantic': round(r.semantic_score, 3), 'why': '; '.join(r.reasons[:2])}
        for r in results
    ])

ravi = next(c for c in candidates if c.id == 'c-001')  # backend engineer
print(ravi.name, '-', ravi.skills)
show(engine.match_jobs_for_candidate(ravi, jobs, k=5))

## Hirer flow — best candidates for a job

In [ ]:
backend_job = next(j for j in jobs if j.id == 'j-001')
print(backend_job.title, '@', backend_job.company)
show(engine.match_candidates_for_job(backend_job, candidates, k=5))

## Next steps

- `python -m mindbridge.cli train` to fit the XGBoost reranker; the engine then uses it automatically.
- Add live jobs via the Adzuna API (`.env` keys) — `load_jobs(sources=['sample','api'])`.
- Later milestones wrap this engine in a FastAPI backend + React hirer/hiree UIs.